# HF LLM: Natural Language → SQL (schema + examples)

This notebook:
1. Opens the local SQLite database `meta_ads.db`
2. Displays the DB schema (tables + columns)
3. Uses a Hugging Face Inference API LLM (Qwen Instruct by default) to convert natural-language questions into SQLite `SELECT` queries

This demo **generates SQL only** (it does not execute SQL).


## 1) Setup

This section loads configuration and dependencies.

- Set `HF_TOKEN` in your environment (required)
- Optionally set `HF_MODEL_ID` (defaults to a Qwen Instruct model)


In [1]:
import os
print("HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))
print("HF_TOKEN prefix:", os.environ.get("HF_TOKEN")[:8])

HF_TOKEN present: True
HF_TOKEN prefix: hf_Fgwth


In [2]:
import json
import os
import re
import sqlite3
import urllib.request
from pathlib import Path
from textwrap import indent

DB_PATH = Path("meta_ads.db")
assert DB_PATH.exists(), f"DB not found at {DB_PATH.resolve()}"

HF_TOKEN = os.environ.get("HF_TOKEN")
HF_MODEL_ID = os.environ.get("HF_MODEL_ID") or "openai/whisper-large-v3"

if not HF_TOKEN:
    raise RuntimeError(
        "Missing HF_TOKEN. Set it in your environment, then restart the kernel."
    )

print("Using model:", HF_MODEL_ID)


Using model: openai/whisper-large-v3


## 2) Connect to SQLite and display schema

This section connects to `meta_ads.db` and prints a compact schema string.
We will feed this schema into the LLM so it uses the correct table/column names.


In [3]:
conn = sqlite3.connect(DB_PATH)

def get_tables(conn: sqlite3.Connection):
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    return [r[0] for r in rows]

def get_table_info(conn: sqlite3.Connection, table: str):
    # PRAGMA table_info returns: cid, name, type, notnull, dflt_value, pk
    return conn.execute(f"PRAGMA table_info({table})").fetchall()

def build_schema_string(conn: sqlite3.Connection) -> str:
    lines = []
    for t in get_tables(conn):
        cols = get_table_info(conn, t)
        col_str = ", ".join([f"{c[1]} {c[2]}" for c in cols])
        lines.append(f"TABLE {t}({col_str})")
    return "\n".join(lines)

SCHEMA = build_schema_string(conn)
print(SCHEMA)


TABLE ad_products(product_id INTEGER, product_name TEXT, placement TEXT, objective TEXT, pricing_model TEXT, active_from DATE)
TABLE ad_transactions(transaction_id INTEGER, event_ts TEXT, advertiser_id INTEGER, product_id INTEGER, campaign_id TEXT, currency TEXT, impressions INTEGER, clicks INTEGER, conversions INTEGER, spend_usd REAL, revenue_usd REAL)
TABLE advertisers(advertiser_id INTEGER, advertiser_name TEXT, industry TEXT, hq_country TEXT, account_tier TEXT, created_at DATE)


## 3) Hugging Face Inference API client

This section defines a tiny helper that sends a prompt to the Hugging Face Inference API and returns the model’s generated text.

We keep dependencies minimal by using Python’s built-in `urllib`.


In [4]:
def hf_generate(prompt: str, max_new_tokens: int = 256, temperature: float = 0.0) -> str:
    """Call HF via InferenceClient using an OpenAI-compatible chat.completions API.

    This mirrors the working pattern from test_llm.ipynb.
    """
    messages = [{"role": "user", "content": prompt}]

    response = client.chat.completions.create(
        model=HF_MODEL_ID,
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=temperature,
    )

    # Extract the text content from the first choice
    content = response.choices[0].message.content
    # Some clients return a list of parts; handle both cases defensively
    if isinstance(content, list):
        return "".join(part["text"] if isinstance(part, dict) and "text" in part else str(part) for part in content)
    return str(content)


## 4) Prompt template for SQLite SQL

This section builds the prompt we send to the LLM. The key “prompt optimization” ideas here are:

- Provide the **schema** so the model knows the correct tables/columns
- Add **constraints** (SQLite, single SELECT, no markdown fences)
- Add **few-shot examples** to steer formatting and joins


In [5]:
FEW_SHOT = """
Example 1
Question: What is total revenue?
SQL: SELECT SUM(revenue_usd) AS total_revenue_usd FROM ad_transactions;

Example 2
Question: Revenue by placement
SQL: SELECT p.placement, SUM(t.revenue_usd) AS revenue_usd
     FROM ad_transactions t
     JOIN ad_products p ON p.product_id = t.product_id
     GROUP BY p.placement
     ORDER BY revenue_usd DESC;
""".strip()

PROMPT_TEMPLATE = """
You are an expert data analyst.
Given a SQLite database schema, write ONE SQLite SQL query that answers the question.

Rules:
- Output ONLY the SQL query.
- Must be a single SELECT statement.
- Do NOT use ``` fences.
- Use only table/column names that exist in the schema.

Schema:
{schema}

{few_shot}

Question: {question}
SQL:
""".strip()

def build_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=SCHEMA, few_shot=FEW_SHOT, question=question)


## 5) Natural-language → SQL examples

This section runs a few example questions (easy → hard) through the LLM and prints the generated SQL.

Tip: If the model outputs extra text, we’ll extract the first `SELECT ...` block.


In [6]:
def extract_first_select(text: str) -> str:
    m = re.search(r"\bselect\b[\s\S]*", text, flags=re.IGNORECASE)
    if not m:
        return text.strip()
    sql = m.group(0).strip()
    sql = re.sub(r"```[\s\S]*?```", "", sql).strip()
    return sql

questions = [
    "What is total revenue?",
    "What is overall ROAS?",
    "Top 10 advertisers by revenue",
    "Revenue by placement",
    "Daily revenue last 30 days",
    "Top 5 products by ROAS last 60 days",
    "Revenue by industry for Enterprise advertisers last 90 days",
    "Top 10 advertisers by revenue for placement = Reels last 45 days"
]

for q in questions:
    prompt = build_prompt(q)
    raw = hf_generate(prompt, max_new_tokens=256, temperature=0.0)
    sql = extract_first_select(raw)
    print("NL:", q)
    print("SQL:\n" + indent(sql, "  "))
    print("-" * 80)


HTTPError: HTTP Error 410: Gone

## 6) Cleanup

Close the SQLite connection.


In [ ]:
conn.close()
